# 🧼 Data Cleaning Phase

This notebook prepares the raw meningioma dataset for analysis.

The goal is to turn messy clinical/radiological data into a clean, consistent table that can later be used for:

- exploratory analysis
- statistical testing
- logistic regression
- model validation
- Streamlit calculator deployment

At this stage, the priority is **data trustworthiness**, not modeling.

No prediction model is built here.  
This phase only creates the clean foundation.

In [1]:
from IPython.display import display
import pandas as pd
from pandas.api.types import is_categorical_dtype
import numpy as np
from datetime import date, datetime
import matplotlib.pyplot as plt
import seaborn as sns


from pathlib import Path

In [2]:
#🟧🟧🟧 load dataset and define the ID col
df = pd.read_excel(
    "data/Meningiomas PSKUS grants.xlsx",
    index_col="Nr."
    )

df.index.name = "case_id"

df.head(0)

,Personas kods,Unnamed: 2,"Vecums, gadi","Dzimums, 0 - vīrietis\n1 - sieviete""","Histoloģija, 0 - nav\n1 - ir","WHO pakāpe (2021), 1 / 2 / 3","Progesterons, 0 - negatīvs\n1 - pozitīvs","Ki-67 (%), skaitlis, %","Smadzeņu parenhīmas invāzija, 0 - nav\n1 - ir","Nekroze histoloģiski, 0 - nav\n1 - ir",...,"Audzēja nekroze, 0 - nav\n1 - ir","Hemorāģiskas sastāvdaļas, 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams","Kaule hiperostoze, 0 - nav\n1 - ir","Kaula invāzija (cortical destruction), 0 - nav\n1 - ir","Tumor Hyperintensity on DWI, 0 - nav\n1 - ir","Tumor Hyperintensity on T2, 0 - nav\n1 - ir","Tumor Hypointensity on T1, 0 - nav\n1 - ir","Sīnuss, 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug","Cauraug falx cerebri 0 - nav, 1 - ir",ADC map value
case_id,,,,,,,,,,,,,,,,,,,,,


In [3]:
#🟧🟧🟧 comfort renaming
COLUMN_RENAME_MAP = {
    "Personas kods": "patient_code",
    "Unnamed: 2": "entry_year",

    "Vecums, gadi": "age",
    'Dzimums, 0 - vīrietis\n1 - sieviete"': "sex",
    "Histoloģija, 0 - nav\n1 - ir": "histology_available",
    "WHO pakāpe (2021), 1 / 2 / 3": "who_grade",
    "Progesterons, 0 - negatīvs\n1 - pozitīvs": "progesterone_pos",
    "Ki-67 (%), skaitlis, %": "ki67_pct",

    "Smadzeņu parenhīmas invāzija, 0 - nav\n1 - ir": "brain_invasion",
    "Nekroze histoloģiski, 0 - nav\n1 - ir": "hist_necrosis",

    "MRI izmeklējuma datums": "mri_date",
    "Puse, 1 - labā\n2 - kreisā\n3 - viduslīnija": "side",
    "Lokalizācija: skull base / non–skull base, 0 - non-skull base\n1 - skull base": "tumor_location",
    "Cik meningiomas?": "meningioma_count",

    "Max diametrs, skaitlis,cm": "max_diameter_cm",
    "Tilpums": "tumor_volume",
    "Pamatmodalitāte analīzei, 0 - MRI\n1 - CT\n3 - MRI+CT": "base_modality",
    "K/v i/v, 0 - nav\n1 - ir": "iv_contrast",
    "0 - primārs\n1 - recidīvs": "tumor_episode",

    "Audzēja robeža, 1 = gluda, \n2 = neregulāra": "tumor_margin",
    "Dural tail sign, 0 - nav\n1 - ir": "dural_tail",
    "Gredzenveida kontrastēšanās (tumor capsular enhancement), 0 - nav\n1 - ir": "capsular_enhancement",
    "Kontrastēšanās veids, 0 - homogēna\n1 - heterogēna": "heterogeneous_enhancement",

    "Perifokāla tūska, 0 - nav\n1 - ir": "perifocal_edema",
    "Perifokālas tūskas tilpums, cm2": "edema_volume_cm3",
    "Masas efekts, 0 - nav\n1 - ir": "mass_effect",

    "Audzēja kalcifikācija, 0 - nav\n1 - ir": "calcification",
    "Cistiskas komponentes, 0 - nav\n1 - ir": "cystic_component",
    "Audzēja nekroze, 0 - nav\n1 - ir": "necrosis",
    "Hemorāģiskas sastāvdaļas, 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams ": "hemorrhage",

    "Kaule hiperostoze, 0 - nav\n1 - ir": "hyperostosis",
    "Kaula invāzija (cortical destruction), 0 - nav\n1 - ir": "cortical_destruction",

    "Tumor Hyperintensity on DWI, 0 - nav\n1 - ir": "dwi_hyperintensity",
    "Tumor Hyperintensity on T2, 0 - nav\n1 - ir": "t2_hyperintensity",
    "Tumor Hypointensity on T1, 0 - nav\n1 - ir": "t1_hypointensity",

    "Sīnuss, 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug": "sinus_invasion",
    "Cauraug falx cerebri 0 - nav, 1 - ir": "transfalcine_extension",
    "ADC map value": "adc_value",
}

df = df.rename(columns=COLUMN_RENAME_MAP)

df.columns

Index(['patient_code', 'entry_year', 'age', 'sex', 'histology_available',
       'who_grade', 'progesterone_pos', 'ki67_pct', 'brain_invasion',
       'hist_necrosis', 'mri_date', 'side', 'tumor_location',
       'meningioma_count', 'max_diameter_cm', 'tumor_volume', 'base_modality',
       'iv_contrast', 'tumor_episode', 'tumor_margin', 'dural_tail',
       'capsular_enhancement', 'heterogeneous_enhancement', 'perifocal_edema',
       'edema_volume_cm3', 'mass_effect', 'calcification', 'cystic_component',
       'necrosis', 'hemorrhage', 'hyperostosis', 'cortical_destruction',
       'dwi_hyperintensity', 't2_hyperintensity', 't1_hypointensity',
       'sinus_invasion', 'transfalcine_extension', 'adc_value'],
      dtype='str')

In [4]:
#🟧🟧🟧 drop unneeded columns
df = df.drop(columns=["patient_code", "entry_year", "mri_date"])

# check for sure that patient_code is not in df
assert "patient_code" not in df.columns

In [5]:
#🟧🟧🟧 hotfix for sigita columns
def fix_sigita_to_numeric(x):
    if isinstance(x, (pd.Timestamp, date, datetime)):
        return float(f"{x.day}.{x.month}")
    if isinstance(x, str) and "," in x:
        return x.replace(",", ".")
    return x

really_numeric = ['age', 'sex', 'histology_available', 'who_grade', 'progesterone_pos',
       'brain_invasion', 'hist_necrosis', 'side', 'tumor_location',
       'meningioma_count', 'max_diameter_cm', 'tumor_volume', 'base_modality',
       'iv_contrast', 'tumor_episode', 'tumor_margin',
       'dural_tail', 'capsular_enhancement', 'heterogeneous_enhancement',
       'perifocal_edema', 'edema_volume_cm3', 'mass_effect', 'calcification',
       'cystic_component', 'necrosis', 'hemorrhage', 'hyperostosis',
       'cortical_destruction', 'dwi_hyperintensity', 't2_hyperintensity',
       't1_hypointensity', 'sinus_invasion', 'transfalcine_extension',
       'adc_value']

for col in really_numeric:
    raw_col = f"{col}_raw"
    df[raw_col] = df[col]
    print(f"{raw_col} - {df[raw_col].dtype} - {df[raw_col].isna().sum()}")
    
    cleaned = df[col].apply(fix_sigita_to_numeric)

    df[col] = pd.to_numeric(cleaned, errors="coerce")

    display(
        df.loc[df[col].isna() & df[raw_col].notna(), [raw_col, col]]
        .drop_duplicates()
        )
    print(f"{col} - {df[col].dtype} - {df[col].isna().sum()}")
    print("=" * 40)

df = df.drop(columns=[col for col in df.columns if "_raw" in col])

age_raw - float64 - 1


,age_raw,age
case_id,,


age - float64 - 1
sex_raw - float64 - 1


,sex_raw,sex
case_id,,


sex - float64 - 1
histology_available_raw - float64 - 1


...

sinus_invasion - float64 - 183
transfalcine_extension_raw - float64 - 390


,transfalcine_extension_raw,transfalcine_extension
case_id,,


transfalcine_extension - float64 - 390
adc_value_raw - object - 162


,adc_value_raw,adc_value
case_id,,
16,NAV ADC,NaN
30,ADC neredz meningiomu,NaN
47,nosūtīts,NaN
113,nav adc,NaN
133,nav mri,NaN
395,nav,NaN


adc_value - float64 - 178


In [6]:
#🟧🟧🟧 fix string being datetime (ki67_pct)
def fix_ki67_being_datetime(x):
    if isinstance(x, (pd.Timestamp, date, datetime)):
        return f"{x.day}-{x.month}"
    return x

ki_like_cols = ["ki67_pct"]

for col in ki_like_cols:
    raw_col = f"{col}_raw"
    df[raw_col] = df[col]
    print(f"{raw_col} - {df[raw_col].dtype} - {df[raw_col].isna().sum()}")

    date_mask = df[col].apply(lambda x: isinstance(x, (pd.Timestamp, date, datetime)))

    df[col] = df[col].apply(fix_ki67_being_datetime).astype("string")

    display(df.loc[date_mask, [raw_col, col]])
    print(f"{col} - {df[col].dtype} - {df[col].isna().sum()}")

df = df.drop(columns=[col for col in df.columns if "_raw" in col])

ki67_pct_raw - object - 32


,ki67_pct_raw,ki67_pct
case_id,,
2,2025-03-01 00:00:00,1-3
3,2025-05-01 00:00:00,1-5
5,2025-02-01 00:00:00,1-2
6,2025-02-01 00:00:00,1-2
8,2025-02-01 00:00:00,1-2
...,...,...
395,2025-05-03 00:00:00,3-5
396,2025-06-05 00:00:00,5-6
397,2025-07-03 00:00:00,3-7


ki67_pct - string - 32


In [7]:
#🟧🟧🟧 Value Counts on all the columns
for col in df.columns:
    display(df[col].value_counts())

#- histology_available = 2.0 .......53....... (jo 0=nav, 1=ir)
#- tumor_location = 2.0 ......291....... (jo 0=non-skull-base, 1=skull-base)
#- tumor_episode = 2 .....111...... (jo 0=primars, 1=recidivs)
#- tumor_episode = multiplas .....157...... (jo 0=primars, 1=recidivs)
#- tumor_margin = 0 ......87,244,245....... (jo 1=reg, 2=nonreg)
#- hemorrhage = 2.0 ........121,169......... (jo 0=nav, 1=ir). Raksta 2=nav skaidri izvertejams (likt NaN?)
# =========== ALL COMMENTED in the Excel ===========

age
66.0    15
71.0    15
68.0    13
63.0    12
70.0    12
        ..
90.0     1
95.0     1
38.0     1
25.0     1
37.0     1
Name: count, Length: 63, dtype: int64

sex
1.0    276
0.0    121
Name: count, dtype: int64

histology_available
1.0    364
0.0     32
2.0      1
Name: count, dtype: int64

who_grade
1.0    256
2.0     99
3.0     11
Name: count, dtype: int64

progesterone_pos
1.0    355
0.0      9
2.0      1
Name: count, dtype: int64

...

t2_hyperintensity
1.0    191
0.0     31
Name: count, dtype: int64

t1_hypointensity
1.0    209
0.0     16
Name: count, dtype: int64

sinus_invasion
0.0    155
1.0     43
2.0     17
Name: count, dtype: int64

transfalcine_extension
0.0    6
1.0    2
Name: count, dtype: int64

adc_value
0.79    11
0.81     9
0.84     9
0.86     9
0.85     8
        ..
1.11     1
0.65     1
0.97     1
0.62     1
1.38     1
Name: count, Length: 69, dtype: int64

In [8]:
#🟧🟧🟧 Boolean - Value Renaming and Dtyping
bool_cols = [
    'histology_available', 'progesterone_pos',
    'brain_invasion', 'hist_necrosis',
    'iv_contrast', 'dural_tail', 'capsular_enhancement', 'heterogeneous_enhancement',
    'perifocal_edema', 'mass_effect', 'calcification',
    'cystic_component', 'necrosis', 'hemorrhage', 'hyperostosis',
    'cortical_destruction', 'dwi_hyperintensity', 't2_hyperintensity',
    't1_hypointensity', 'transfalcine_extension',]

for col in bool_cols: print(f"{col} - {df[col].dtype} - values:{df[col].nunique()}")

df.histology_available = df.histology_available.replace(2, pd.NA)
df.hemorrhage = df.hemorrhage.replace(2, pd.NA)
df.progesterone_pos = df.progesterone_pos.replace(2, pd.NA)

#⏹️⏹️⏹️ VIEWER
display(df.head(2))
for col in bool_cols:
    vals = set(df[col].dropna().unique())
    assert vals <= {0, 1}, f"{col}: bad values {vals}"

df[bool_cols] = df[bool_cols].astype('boolean')

df.info()


histology_available - float64 - values:3
progesterone_pos - float64 - values:3
brain_invasion - float64 - values:2
hist_necrosis - float64 - values:2
iv_contrast - float64 - values:2
dural_tail - float64 - values:2
capsular_enhancement - float64 - values:2
heterogeneous_enhancement - float64 - values:2
perifocal_edema - float64 - values:2
mass_effect - float64 - values:2
calcification - float64 - values:2
cystic_component - float64 - values:2
necrosis - float64 - values:2
hemorrhage - float64 - values:3
hyperostosis - float64 - values:2
cortical_destruction - float64 - values:2
dwi_hyperintensity - float64 - values:2
t2_hyperintensity - float64 - values:2
t1_hypointensity - float64 - values:2
transfalcine_extension - float64 - values:2


,age,sex,histology_available,who_grade,progesterone_pos,ki67_pct,brain_invasion,hist_necrosis,side,tumor_location,...,necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value
case_id,,,,,,,,,,,,,,,,,,,,,
1,67.0,1.0,0.0,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,67.0,1.0,1.0,1.0,1.0,1-3,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,NaN,0.88


<class 'pandas.DataFrame'>
Index: 398 entries, 1 to 400
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        397 non-null    float64
 1   sex                        397 non-null    float64
 2   histology_available        396 non-null    boolean
 3   who_grade                  366 non-null    float64
 4   progesterone_pos           364 non-null    boolean
 5   ki67_pct                   366 non-null    string 
 6   brain_invasion             366 non-null    boolean
 7   hist_necrosis              366 non-null    boolean
 8   side                       242 non-null    float64
 9   tumor_location             236 non-null    float64
 10  meningioma_count           304 non-null    float64
 11  max_diameter_cm            237 non-null    float64
 12  tumor_volume               295 non-null    float64
 13  base_modality              240 non-null    float64
 14  iv_contras

In [9]:
#🟧🟧🟧 Deriving Categorical Columns

numerical_cols = [
    'adc_value', 'age', 'edema_volume_cm3',
    'max_diameter_cm', 'meningioma_count', 'tumor_volume',]
cols_left_to_dtype = set(df.columns) - set(bool_cols) - set(numerical_cols)
print(f"Cols to Dtype: {cols_left_to_dtype}")

# ✅ who_grade
df.who_grade = pd.Categorical(df.who_grade, categories=[1,2,3], ordered=True)

# ✅ base_modality
df.base_modality = df.base_modality.replace({
    0: "mri",
    1: "ct",
    3: "mri_ct"
    })
df.base_modality = pd.Categorical(df.base_modality, categories=["mri", "ct", "mri_ct"], ordered=False)

# ✅ sex
df.sex = df.sex.replace({
    0: "male",
    1: "female"
    })
df.sex = pd.Categorical(df.sex, categories=["male", "female"], ordered=False)

# ✅ tumor_location
df.tumor_location = df.tumor_location.replace({
    0: "non_skull_base",
    1: "skull_base",
    2: pd.NA
    })
df.tumor_location = pd.Categorical(df.tumor_location, categories=["non_skull_base", "skull_base"], ordered=False)

# ✅ tumor_episode
df.tumor_episode = df.tumor_episode.replace({
    0: "primary",
    1: "recurrent",
    2: pd.NA,
    "multiplas": pd.NA,
    })
df.tumor_episode = pd.Categorical(df.tumor_episode, categories=["primary", "recurrent"], ordered=False)

# ✅ tumor_margin
df.tumor_margin = df.tumor_margin.replace({
    0: pd.NA,
    1: "regular",
    2: "irregular"
    })
df.tumor_margin = pd.Categorical(df.tumor_margin, categories=["regular", "irregular"], ordered=False)

# ✅ sinus_invasion
"Sīnuss, 0 - neieaug, 1 - ieaug, 2 - ieaug un cauraug"
df.sinus_invasion = df.sinus_invasion.replace({
    0: "no_invasion",
    1: "sinus_invasion",
    2: "transsinus_extension"})
df.sinus_invasion = pd.Categorical(df.sinus_invasion, categories=["no_invasion", "sinus_invasion", "transsinus_extension"], ordered=True)

# ✅ side
"Puse, 1 - labā, 2 - kreisā, 3 - viduslīnija"
df.side = df.side.replace({1: "right", 2: "left", 3: "midline"})
df.side = pd.Categorical(df.side, categories=["right", "left", "midline"], ordered=False)

for col in df.columns:
    if is_categorical_dtype(df[col]):
        print(f"=" * 30)
        print(df[col].unique())
        print(f"=" * 30)

cols_done = ["tumor_episode", "tumor_location", "sex", "tumor_margin", "base_modality", "who_grade", "side", "sinus_invasion"]
f"Cols left to dtype: {cols_left_to_dtype - set(cols_done)}"

Cols to Dtype: {'base_modality', 'side', 'ki67_pct', 'tumor_margin', 'sex', 'who_grade', 'sinus_invasion', 'tumor_episode', 'tumor_location'}
['female', 'male', NaN]
Categories (2, str): ['male', 'female']
[NaN, 1, 2, 3]
Categories (3, int64): [1 < 2 < 3]
[NaN, 'right', 'midline', 'left']
Categories (3, str): ['right', 'left', 'midline']
[NaN, 'skull_base', 'non_skull_base']
Categories (2, str): ['non_skull_base', 'skull_base']
[NaN, 'mri', 'mri_ct', 'ct']
Categories (3, str): ['mri', 'ct', 'mri_ct']
[NaN, 'primary', 'recurrent']
Categories (2, str): ['primary', 'recurrent']
[NaN, 'regular', 'irregular']
Categories (2, str): ['regular', 'irregular']
[NaN, 'no_invasion', 'sinus_invasion', 'transsinus_extension']
Categories (3, str): ['no_invasion' < 'sinus_invasion' < 'transsinus_extension']


/var/folders/c6/b51bn8410z541s8nqff2nxcm0000gn/T/ipykernel_21825/3310457055.py:66: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if is_categorical_dtype(df[col]):
/var/folders/c6/b51bn8410z541s8nqff2nxcm0000gn/T/ipykernel_21825/3310457055.py:66: Pandas4Warning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if is_categorical_dtype(df[col]):


"Cols left to dtype: {'ki67_pct'}"

In [10]:
#⏹️⏹️⏹️ VIEWER
display(df.head(2))
display(df.info())


,age,sex,histology_available,who_grade,progesterone_pos,ki67_pct,brain_invasion,hist_necrosis,side,tumor_location,...,necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value
case_id,,,,,,,,,,,,,,,,,,,,,
1,67.0,female,False,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN
2,67.0,female,True,1,True,1-3,False,False,right,skull_base,...,False,False,False,False,True,True,True,no_invasion,<NA>,0.88


<class 'pandas.DataFrame'>
Index: 398 entries, 1 to 400
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   age                        397 non-null    float64 
 1   sex                        397 non-null    category
 2   histology_available        396 non-null    boolean 
 3   who_grade                  366 non-null    category
 4   progesterone_pos           364 non-null    boolean 
 5   ki67_pct                   366 non-null    string  
 6   brain_invasion             366 non-null    boolean 
 7   hist_necrosis              366 non-null    boolean 
 8   side                       242 non-null    category
 9   tumor_location             235 non-null    category
 10  meningioma_count           304 non-null    float64 
 11  max_diameter_cm            237 non-null    float64 
 12  tumor_volume               295 non-null    float64 
 13  base_modality              240 non-null    category

None

In [11]:
#🟧🟧🟧 Deriving Bins (TODO: ask radiologists what is the better division based on regular patient groups)

# ki67_pct (TODO: this is the most important one!!!)
def ki67_midpoint(x):
    if pd.isna(x):
        return pd.NA

    parts = str(x).replace(",", ".").split("-")
    nums = [float(p) for p in parts]

    return sum(nums) / len(nums)

df["ki67_mid"] = df["ki67_pct"].map(ki67_midpoint).astype("Float64")
def ki67_group(x):
    if pd.isna(x):
        return pd.NA
    if x <= 4:
        return "low_le_4"
    if x < 10:
        return "intermediate_5_9"
    return "high_ge_10"

df["ki67_group"] = df["ki67_mid"].map(ki67_group)

df["ki67_group"] = pd.Categorical(
    df["ki67_group"],
    categories=["low_le_4", "intermediate_5_9", "high_ge_10"],
    ordered=True,)
df = df.drop(columns=["ki67_pct", "ki67_mid"])


# ============================================================
# Numeric binning placeholders
# Edit bins/labels when Balodis or data distribution decides.
# ============================================================

def make_bins(series, bins, labels):
    return pd.cut(
        series,
        bins=bins,
        labels=labels,
        include_lowest=True,
        right=True,
        ordered=True,
    )


# adc_value
# df["adc_group"] = make_bins(
#     df["adc_value"],
#     bins=[-float("inf"), 700, 900, float("inf")],
#     labels=["low_adc", "intermediate_adc", "high_adc"],
# )


# age
# df["age_group"] = make_bins(
#     df["age"],
#     bins=[-float("inf"), 49, 64, 74, float("inf")],
#     labels=["age_le_49", "age_50_64", "age_65_74", "age_ge_75"],
# )


# edema_volume_cm3
# NOTE: check this name. Volume should probably be cm3, not cm2.
# df["edema_volume_group"] = make_bins(
#     df["edema_volume_cm3"],
#     bins=[-float("inf"), 0, 10, 50, float("inf")],
#     labels=["no_edema", "small_edema", "moderate_edema", "large_edema"],
# )


# max_diameter_cm
# df["max_diameter_group"] = make_bins(
#     df["max_diameter_cm"],
#     bins=[-float("inf"), 2, 4, 6, float("inf")],
#     labels=["small_le_2cm", "medium_2_4cm", "large_4_6cm", "very_large_gt_6cm"],
# )


# meningioma_count
# df["meningioma_count_group"] = make_bins(
#     df["meningioma_count"],
#     bins=[-float("inf"), 1, float("inf")],
#     labels=["single", "multiple"],
# )


# tumor_volume
# df["tumor_volume_group"] = make_bins(
#     df["tumor_volume"],
#     bins=[-float("inf"), 10, 50, 100, float("inf")],
#     labels=["small_volume", "medium_volume", "large_volume", "very_large_volume"],
# )

df.columns

Index(['age', 'sex', 'histology_available', 'who_grade', 'progesterone_pos',
       'brain_invasion', 'hist_necrosis', 'side', 'tumor_location',
       'meningioma_count', 'max_diameter_cm', 'tumor_volume', 'base_modality',
       'iv_contrast', 'tumor_episode', 'tumor_margin', 'dural_tail',
       'capsular_enhancement', 'heterogeneous_enhancement', 'perifocal_edema',
       'edema_volume_cm3', 'mass_effect', 'calcification', 'cystic_component',
       'necrosis', 'hemorrhage', 'hyperostosis', 'cortical_destruction',
       'dwi_hyperintensity', 't2_hyperintensity', 't1_hypointensity',
       'sinus_invasion', 'transfalcine_extension', 'adc_value', 'ki67_group'],
      dtype='str')

In [12]:
#🟧🟧🟧 FINISHING
df.info()
df.isna().sum().sort_values(ascending=False)
df.nunique().sort_values()

<class 'pandas.DataFrame'>
Index: 398 entries, 1 to 400
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   age                        397 non-null    float64 
 1   sex                        397 non-null    category
 2   histology_available        396 non-null    boolean 
 3   who_grade                  366 non-null    category
 4   progesterone_pos           364 non-null    boolean 
 5   brain_invasion             366 non-null    boolean 
 6   hist_necrosis              366 non-null    boolean 
 7   side                       242 non-null    category
 8   tumor_location             235 non-null    category
 9   meningioma_count           304 non-null    float64 
 10  max_diameter_cm            237 non-null    float64 
 11  tumor_volume               295 non-null    float64 
 12  base_modality              240 non-null    category
 13  iv_contrast                262 non-null    boolean 

capsular_enhancement           2
calcification                  2
mass_effect                    2
hemorrhage                     2
perifocal_edema                2
heterogeneous_enhancement      2
hyperostosis                   2
dural_tail                     2
tumor_margin                   2
tumor_episode                  2
iv_contrast                    2
cystic_component               2
cortical_destruction           2
t2_hyperintensity              2
t1_hypointensity               2
tumor_location                 2
hist_necrosis                  2
brain_invasion                 2
progesterone_pos               2
transfalcine_extension         2
histology_available            2
sex                            2
dwi_hyperintensity             2
necrosis                       2
sinus_invasion                 3
ki67_group                     3
base_modality                  3
side                           3
who_grade                      3
meningioma_count               4
edema_volu

In [13]:
df.to_excel("data/cleaned/meningiomas_cleaned.xlsx", index=False)
df.to_csv("data/cleaned/meningiomas_cleaned.csv", index=False)

# **ULTIMATE OVERVIEW**